# Sorting 128 Random Integers
Bubble Sort • Insertion Sort • Merge Sort • Quick Sort

Run the setup cell first, then run any algorithm cell independently. Each uses a fresh copy of the **same seeded random array**. Press ▶ below each animation to play; pause or step through frames to inspect it. Rendering may take a little time.

The animations use 150 ms per frame and at most 500 frames each. All sorting operations are executed and counted, but evenly spaced snapshots are displayed when there are too many operations. **Playback duration is not an execution-time benchmark.** Increase `MAX_ANIMATION_FRAMES` to see more individual operations.

The short reflections below are draft observations to review after watching the animations.

## Shared setup and animation helper

In [8]:
import random
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Patch
from IPython.display import HTML, display

N = 128                         # Change to 256 if desired.
SEED = 42                       # Reproducible input for all four algorithms.
ANIMATION_INTERVAL = 150         # Milliseconds between displayed frames.
MAX_ANIMATION_FRAMES = 500       # Limits embedded animation size.
plt.rcParams["animation.embed_limit"] = 200  # MB per embedded animation.

rng = random.Random(SEED)
INITIAL_ARRAY = [rng.randint(1, 1000) for _ in range(N)]

COLORS = {
    "idle": "#9CA3AF", "active": "#60A5FA", "compare": "#F59E0B",
    "move": "#EF4444", "pivot": "#A855F7", "done": "#22C55E"
}

def snapshot(a, message, comparisons, active=(), compared=(), moved=(),
             done=(), pivot=None):
    """Copy a state so later array changes do not alter this frame."""
    colors = [COLORS["idle"]] * len(a)
    for indices, role in [(done, "done"), (active, "active"),
                          (compared, "compare"), (moved, "move")]:
        for i in indices:
            colors[i] = COLORS[role]
    if pivot is not None:
        colors[pivot] = COLORS["pivot"]
    return a.copy(), colors, message, comparisons

def animate_sort(sort_steps, name):
    """Run twice: count events, then retain at most 500 full snapshots.

    The algorithms are deterministic for a fixed input. Two passes keep
    storage bounded without skipping any actual sorting operations.
    """
    if MAX_ANIMATION_FRAMES < 2:
        raise ValueError("MAX_ANIMATION_FRAMES must be at least 2")
    total = sum(1 for _ in sort_steps(INITIAL_ARRAY.copy()))
    count = min(total, MAX_ANIMATION_FRAMES)
    selected = {i * (total - 1) // (count - 1) for i in range(count)}
    frames = [(step, state) for step, state in
              enumerate(sort_steps(INITIAL_ARRAY.copy())) if step in selected]

    fig, ax = plt.subplots(figsize=(11, 4.8), dpi=80)
    bars = ax.bar(range(N), INITIAL_ARRAY, width=0.9, color=COLORS["idle"])
    ax.set(xlim=(-1, N), ylim=(0, max(INITIAL_ARRAY) * 1.08),
           xlabel="Array index", ylabel="Integer value")
    ax.set_title(name, fontsize=17, weight="bold", pad=42)
    status = ax.text(0, 1.025, "", transform=ax.transAxes, fontsize=9)
    legend = [("idle", "Other"), ("active", "Active range"),
              ("compare", "Comparison"), ("move", "Move / write / swap"),
              ("pivot", "Pivot"), ("done", "Sorted / processed*")]
    fig.legend(handles=[Patch(color=COLORS[k], label=v) for k, v in legend],
               loc="lower center", ncol=3, fontsize=9, frameon=False)
    fig.subplots_adjust(top=0.76, bottom=0.24, left=0.07, right=0.98)

    def update(frame_number):
        step, (values, colors, message, comparisons) = frames[frame_number]
        for bar, value, color in zip(bars, values, colors):
            bar.set_height(value)
            bar.set_color(color)
        status.set_text(f"{message}\nKey comparisons: {comparisons:,} | "
                        f"Event {step + 1:,}/{total:,} | "
                        f"Frame {frame_number + 1}/{len(frames)}")
        return (*bars, status)

    animation = FuncAnimation(fig, update, frames=len(frames),
                              interval=ANIMATION_INTERVAL, repeat=False,
                              blit=False)
    plt.close(fig)  # Avoid displaying an extra static plot in Jupyter.
    display(HTML(animation.to_jshtml(default_mode="once")))

## 1. Bubble Sort

In [ ]:
def bubble_steps(a):
    """Compare neighbors; each pass moves a largest remaining value right."""
    n = len(a)
    comparisons = 0
    yield snapshot(a, "Initial array", comparisons)
    for end in range(n - 1, 0, -1):
        swapped = False
        for j in range(end):
            comparisons += 1
            yield snapshot(a, f"Compare neighbors {j} and {j + 1}", comparisons,
                           compared=(j, j + 1), done=range(end + 1, n))
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swapped = True
                yield snapshot(a, f"Swap neighbors {j} and {j + 1}", comparisons,
                               moved=(j, j + 1), done=range(end + 1, n))
        yield snapshot(a, f"Pass complete: index {end} is final", comparisons,
                       done=range(end, n))
        if not swapped:          # No swaps means the remaining prefix is sorted.
            break
    yield snapshot(a, "Finished", comparisons, done=range(n))

animate_sort(bubble_steps, "Bubble Sort")

## 2. Insertion Sort

In [ ]:
def insertion_steps(a):
    """Save a key, shift larger prefix values right, then insert the key."""
    n = len(a)
    comparisons = 0
    yield snapshot(a, "Initial array", comparisons, done=range(min(1, n)))
    for i in range(1, n):
        key = a[i]              # Stored separately while its slot is reused.
        j = i - 1
        yield snapshot(a, f"Save key {key} from index {i}", comparisons,
                       moved=(i,), done=range(i))
        while j >= 0:
            comparisons += 1
            yield snapshot(a, f"Compare index {j} with saved key {key}",
                           comparisons, compared=(j,), active=range(i + 1))
            if a[j] <= key:
                break
            a[j + 1] = a[j]     # Shift right to make room for the saved key.
            yield snapshot(a, f"Shift {j} to {j + 1}; saved key = {key}",
                           comparisons, moved=(j, j + 1), active=range(i + 1))
            j -= 1
        a[j + 1] = key
        yield snapshot(a, f"Insert key {key} at {j + 1}; prefix is sorted",
                       comparisons, moved=(j + 1,), done=range(i + 1))
    yield snapshot(a, "Finished", comparisons, done=range(n))

animate_sort(insertion_steps, "Insertion Sort")

## 3. Merge Sort

In [ ]:
def merge_steps(a):
    """Recursively split ranges, then merge sorted halves using buffers."""
    comparisons = 0
    yield snapshot(a, "Initial array", comparisons)

    def sort(lo, hi, depth):
        nonlocal comparisons
        if hi - lo <= 1:
            return
        mid = (lo + hi) // 2
        yield snapshot(a, f"Depth {depth}: split [{lo}, {hi}) at {mid}",
                       comparisons, active=range(lo, hi))
        yield from sort(lo, mid, depth + 1)
        yield from sort(mid, hi, depth + 1)
        left, right = a[lo:mid], a[mid:hi]  # Temporary sorted halves.
        i = j = 0
        k = lo
        while i < len(left) and j < len(right):
            comparisons += 1
            # These are original buffer-source positions; the array may
            # already have overwritten them. Text shows actual compared values.
            yield snapshot(a, f"Depth {depth}: compare buffers {left[i]} and {right[j]}",
                           comparisons, active=range(lo, hi),
                           compared=(lo + i, mid + j))
            if left[i] <= right[j]:
                a[k] = left[i]
                i += 1
            else:
                a[k] = right[j]
                j += 1
            yield snapshot(a, f"Depth {depth}: merge-write {a[k]} at {k}",
                           comparisons, active=range(lo, hi), moved=(k,))
            k += 1
        # Once one buffer is exhausted, copy the other without key comparisons.
        while i < len(left):
            a[k] = left[i]
            yield snapshot(a, f"Depth {depth}: copy left remainder to {k}",
                           comparisons, active=range(lo, hi), moved=(k,))
            i += 1
            k += 1
        while j < len(right):
            a[k] = right[j]
            yield snapshot(a, f"Depth {depth}: copy right remainder to {k}",
                           comparisons, active=range(lo, hi), moved=(k,))
            j += 1
            k += 1
        yield snapshot(a, f"Depth {depth}: merged [{lo}, {hi})", comparisons,
                       done=range(lo, hi))

    yield from sort(0, len(a), 0)
    yield snapshot(a, "Finished", comparisons, done=range(len(a)))

animate_sort(merge_steps, "Merge Sort")

## 4. Quick Sort

In [ ]:
def quick_steps(a):
    """Lomuto partition: use the last element as pivot, then recurse."""
    comparisons = 0
    final_positions = set()
    yield snapshot(a, "Initial array", comparisons)

    def sort(lo, hi, depth):
        nonlocal comparisons
        if lo > hi:
            return
        if lo == hi:
            final_positions.add(lo)
            yield snapshot(a, f"Depth {depth}: singleton {lo} is final",
                           comparisons, done=final_positions)
            return
        pivot_value = a[hi]
        boundary = lo
        yield snapshot(a, f"Depth {depth}: partition [{lo}, {hi}], pivot {pivot_value}",
                       comparisons, active=range(lo, hi + 1),
                       pivot=hi, done=final_positions)
        for j in range(lo, hi):
            comparisons += 1
            yield snapshot(a, f"Depth {depth}: compare {j} to pivot; boundary {boundary}",
                           comparisons, active=range(lo, hi + 1),
                           compared=(j,), pivot=hi, done=final_positions)
            if a[j] <= pivot_value:
                a[boundary], a[j] = a[j], a[boundary]
                yield snapshot(a, f"Depth {depth}: swap {boundary} and {j}",
                               comparisons, active=range(lo, hi + 1),
                               moved=(boundary, j), pivot=hi, done=final_positions)
                boundary += 1
        a[boundary], a[hi] = a[hi], a[boundary]
        final_positions.add(boundary)
        yield snapshot(a, f"Depth {depth}: place pivot at final index {boundary}",
                       comparisons, moved=(boundary, hi), done=final_positions)
        yield snapshot(a, f"Depth {depth}: partition complete; recurse on each side",
                       comparisons, done=final_positions)
        yield from sort(lo, boundary - 1, depth + 1)
        yield from sort(boundary + 1, hi, depth + 1)

    yield from sort(0, len(a) - 1, 0)
    yield snapshot(a, "Finished", comparisons, done=range(len(a)))

animate_sort(quick_steps, "Quick Sort")